# ELEC 475 Lab 3 - Train CorrectionNetModified on Colab (Step 5)

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

This notebook only does the slow part (training). Afterwards, download `colab_results.zip`, copy the files into your local `lab3/` folder, and run `correct.py` and `evaluate.py` locally.

In [ ]:
# 1. Check the GPU is available
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 2. Get the code from GitHub
%cd /content
!rm -rf ELEC_475_Lab3
!git clone https://github.com/geoffrey329/ELEC_475_Lab3.git
%cd /content/ELEC_475_Lab3
!pip install -q torchinfo

In [ ]:
# 3. Download the Oxford-IIIT Pet images (~800 MB) into data/oxford-iiit-pet-noses/images-original/images
!mkdir -p data/oxford-iiit-pet-noses/images-original
!wget -q --show-progress https://thor.robots.ox.ac.uk/pets/images.tar.gz -O /content/images.tar.gz
!tar -xzf /content/images.tar.gz -C data/oxford-iiit-pet-noses/images-original
!ls data/oxford-iiit-pet-noses/images-original/images | wc -l

In [ ]:
# 4. Upload the label files from your local data/oxford-iiit-pet-noses/ folder
#    (select train_noses.txt and test_noses.txt)
from google.colab import files
import shutil
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'data/oxford-iiit-pet-noses/{name}')
!wc -l data/oxford-iiit-pet-noses/*.txt

In [ ]:
# 5. Check both models build (expect 766,594 and 1,108,642 params)
!python -m models.correction | grep -E 'Testing|Total params|not fully'

In [ ]:
# 6a. Train the modified model WITH augmentation
!python train_correction.py --modified --save-model checkpoints/correction_mod_aug.pth --plot-file plots/loss_correction_mod_aug.png

In [ ]:
# 6b. (Optional) Train the modified model WITHOUT augmentation
#     Skip this if you already have checkpoints/correction_mod_noaug.pth locally.
!python train_correction.py --modified --no-augment --save-model checkpoints/correction_mod_noaug.pth --plot-file plots/loss_correction_mod_noaug.png

In [ ]:
# 7. Zip the trained checkpoints and loss curves, then download them
!zip -j colab_results.zip checkpoints/correction_mod_*.pth plots/loss_correction_mod_*.png
!unzip -l colab_results.zip
files.download('colab_results.zip')